# Notebook 05 — Copy Number Variation (CNV) Inference

**Project:** RetinoblastomaAtlas  
**Input:** `data/processed/05_annotated.h5ad`  
**Output:** `data/processed/06_cnv.h5ad`, CNV heatmap, UMAP, per-cell scores

---

## Scientific Background

Retinoblastoma is initiated by biallelic inactivation of the **RB1 tumour suppressor** (chromosome 13q14), but progression to extraocular disease involves additional somatic copy number alterations (SCNAs). The most consistently reported SCNAs in high-risk RB are:

- **Chromosome 6p amplification** (E2F3, DEK, ID4) — aggressive behaviour and poor differentiation
- **Chromosome 1q gain** (MDM4) — p53 pathway evasion
- **Chromosome 16q loss** — recurrent in metastatic disease
- **Chromosome 2p amplification** (MYCN) — Subtype 2 / MYCN-amplified RB

## Why Infer CNV from scRNA-seq?

1. **Distinguishes tumour from stromal cells** without matched WGS data
2. **Reveals intra-tumoral heterogeneity**: subclones with distinct SCNA profiles may differ in invasive potential
3. **Validates annotation**: tumour cells should show recurrent CNV patterns; immune/glial cells should appear diploid

## Method — Sliding-Window CNV Inference

Implements the **inferCNV approach** (Tirosh et al. 2016, *Science*):
1. Order genes by chromosomal position
2. Compute mean expression in consecutive windows (101 genes)
3. Normalize by expression of reference (non-malignant) cells
4. Clip to [-3, 3] and compute per-cell CNV load (mean |deviation|)

**References:**  
- Tirosh I et al. *Science* 2016. https://doi.org/10.1126/science.aad0501  
- Gao R et al. *Nat Biotechnol* 2021. https://doi.org/10.1038/s41587-020-00795-2

## Parameters

In [ ]:
# --- Reference cells (assumed diploid, used to normalize CNV signal) ---
REFERENCE_CELL_TYPES = [
    "Muller_glia",
    "Microglia_TAM",
    "Endothelial",
    "Astrocyte",
]

# --- RB-specific chromosomal regions ---
RB_SCNA_REGIONS = {
    "chr2p_MYCN":  ("2",  0,           50_000_000),
    "chr6p_E2F3":  ("6",  0,           65_000_000),
    "chr13q_RB1":  ("13", 45_000_000,  70_000_000),
    "chr16q_loss": ("16", 65_000_000, 120_000_000),
    "chr1q_gain":  ("1", 120_000_000, 250_000_000),
}

WINDOW_SIZE = 101  # genes per chromosomal window (standard inferCNV)

## Setup

In [ ]:
import warnings
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1

ROOT     = Path("..").resolve()
IN_H5AD  = ROOT / "data" / "processed" / "05_annotated.h5ad"
OUT_H5AD = ROOT / "data" / "processed" / "06_cnv.h5ad"
FIG_DIR  = ROOT / "results" / "figures"
TAB_DIR  = ROOT / "results" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

## Step 1 — Load Annotated Atlas

In [ ]:
print("Step 1 — Loading annotated atlas")
adata = sc.read_h5ad(IN_H5AD)
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"  Cell types: {adata.obs['cell_type_broad'].value_counts().to_dict()}")

## Step 2 — Fetch Gene Chromosomal Positions

Chromosomal gene order is **fundamental** to CNV inference: amplified regions appear as contiguous blocks of high expression when genes are ordered by genomic position.

The pipeline reads `chromosome` and `start` columns from `adata.var` if available, or falls back to a pre-compiled GRCh38 gene position table.

In [ ]:
print("Step 2 — Fetching gene chromosomal positions")

required_cols = {"chromosome", "start"}
if required_cols.issubset(set(adata.var.columns)):
    print("  Using chromosomal positions from adata.var")
    positions = adata.var[["chromosome", "start"]].copy()
else:
    print("  Attempting to load pre-compiled gene positions...")
    pos_file = ROOT / "data" / "processed" / "gene_positions_grch38.csv"
    if not pos_file.exists():
        raise FileNotFoundError(
            f"{pos_file} not found. Generate gene positions from Ensembl GRCh38 GTF."
        )
    positions = pd.read_csv(pos_file, index_col=0)

valid_chroms = [str(i) for i in range(1, 23)]
positions = positions[positions["chromosome"].astype(str).isin(valid_chroms)]
positions["chromosome"] = positions["chromosome"].astype(int)
positions = positions.sort_values(["chromosome", "start"])
print(f"  Positioned genes across {positions['chromosome'].nunique()} chromosomes")

## Step 3 — Identify Reference (Non-Malignant) Cells

Reference cells are assumed to be **diploid**. Subtracting their per-window mean expression cancels gene-cluster expression patterns that are not copy-number-driven (e.g., HOX gene clusters).

In [ ]:
print("Step 3 — Identifying reference cells")
ref_mask = adata.obs["cell_type_broad"].isin(REFERENCE_CELL_TYPES).values
n_ref = ref_mask.sum()
print(f"  Reference cells: {n_ref:,}")
if n_ref < 100:
    print(f"  WARNING: Only {n_ref} reference cells. CNV normalisation may be unreliable.")
else:
    print(f"  Using {n_ref:,} reference cells for normalization.")

## Step 4 — Compute CNV Matrix

For each window of 101 consecutive genes (sorted by chromosomal position):
- Compute per-cell mean expression from `.layers['scvi_normalized']`
- Subtract per-window mean of reference cells
- Clip result to [-3, 3]

**Result:** cells x windows matrix encoding relative copy number (positive = amplification, negative = deletion).

In [ ]:
print("Step 4 — Computing CNV matrix")

common_genes = positions.index.intersection(adata.var_names)
print(f"  {len(common_genes):,} genes with chromosomal positions")

pos_ordered = positions.loc[common_genes].sort_values(["chromosome", "start"])
gene_order  = pos_ordered.index.tolist()

if "scvi_normalized" in adata.layers:
    expr = adata[:, gene_order].layers["scvi_normalized"]
    print("  Using .layers['scvi_normalized'] (batch-corrected)")
else:
    print("  WARNING: using .X (log-normalized)")
    expr = adata[:, gene_order].X

if hasattr(expr, "toarray"):
    expr = expr.toarray()

n_cells, n_genes = expr.shape
n_windows = n_genes - WINDOW_SIZE + 1
cnv_matrix = np.zeros((n_cells, n_windows), dtype=np.float32)
ref_mean_total = expr[ref_mask].mean(axis=0)

print(f"  Computing {n_windows:,} windows across {n_cells:,} cells...")
for w in range(n_windows):
    window_expr = expr[:, w : w + WINDOW_SIZE].mean(axis=1)
    ref_window  = ref_mean_total[w : w + WINDOW_SIZE].mean()
    cnv_matrix[:, w] = window_expr - ref_window

cnv_matrix = np.clip(cnv_matrix, -3, 3)
print(f"  CNV matrix: {n_cells:,} cells x {n_windows:,} windows")

## Step 5 — Per-Cell CNV Load and Tumour Calling

**CNV load** = mean |CNV score| across all windows.  
**Tumour threshold**: `mean_reference + 3 x SD_reference`

In [ ]:
print("Step 5 — CNV load and tumour calling")

cnv_load = np.mean(np.abs(cnv_matrix), axis=1)
adata.obs["cnv_load"] = cnv_load

ref_load  = cnv_load[ref_mask]
threshold = ref_load.mean() + 3 * ref_load.std()
adata.obs["is_tumour_cnv"] = (cnv_load > threshold).map({True: "Tumour", False: "Normal/uncertain"})

n_tumour = (adata.obs["is_tumour_cnv"] == "Tumour").sum()
print(f"  CNV threshold: {threshold:.4f}")
print(f"  Tumour cells: {n_tumour:,} / {adata.n_obs:,} ({100 * n_tumour / adata.n_obs:.1f}%)")

adata.obsm["X_cnv"] = cnv_matrix.astype(np.float32)

## Step 6 — Region-Specific RB SCNA Scores

| Region | Gene | Significance |
|---|---|---|
| chr2p | MYCN | Subtype 2 amplification |
| chr6p | E2F3/DEK | Aggressive, poor differentiation |
| chr13q | RB1 | Initiating deletion |
| chr16q | — | Metastatic progression |
| chr1q | MDM4 | p53 pathway evasion |

In [ ]:
print("Step 6 — Region-specific SCNA scores")

window_chroms = pos_ordered["chromosome"].values[:n_windows].astype(int)
window_starts = pos_ordered["start"].values[:n_windows]

for region_name, (chrom, start, end) in RB_SCNA_REGIONS.items():
    chrom_int = int(chrom)
    win_mask  = (
        (window_chroms == chrom_int)
        & (window_starts >= start)
        & (window_starts < end)
    )
    if win_mask.sum() > 0:
        region_score = cnv_matrix[:, win_mask].mean(axis=1)
        adata.obs[f"cnv_{region_name}"] = region_score
        print(f"  {region_name}: {win_mask.sum()} windows, mean = {region_score.mean():.4f}")
    else:
        print(f"  WARNING: No windows for {region_name}")

## Step 7 — CNV Heatmap

The **primary CNV QC plot**. Tumour cells should show coherent red/blue blocks at expected chromosomal coordinates; stromal cells should appear grey.

- **Rows**: cells ordered by cell type
- **Columns**: chromosomal windows ordered by genomic position
- **Color**: red = amplification, blue = deletion, grey = neutral

In [ ]:
cell_types = adata.obs["cell_type_broad"]
n_sample   = 2000
idx = np.random.choice(len(cell_types), size=min(n_sample, len(cell_types)), replace=False)
idx = idx[np.argsort(cell_types.iloc[idx].values)]

chrom_changes = pos_ordered["chromosome"].values
chrom_ticks, chrom_labels, prev_chrom = [], [], None
for i, c in enumerate(chrom_changes):
    if c != prev_chrom:
        chrom_ticks.append(i)
        chrom_labels.append(str(c))
        prev_chrom = c

fig, ax = plt.subplots(figsize=(18, 6))
im = ax.imshow(
    cnv_matrix[idx], aspect="auto", cmap="RdBu_r",
    vmin=-1.5, vmax=1.5, interpolation="none",
)
ax.set_xticks(chrom_ticks)
ax.set_xticklabels(chrom_labels, fontsize=7)
ax.set_xlabel("Chromosomal position (windows)", fontsize=10)
ax.set_ylabel(f"Cells (n={len(idx)}, ordered by type)", fontsize=10)
ax.set_title(
    "Inferred Copy Number Variation\n(Red = amplification, Blue = deletion, Grey = neutral)",
    fontsize=11, fontweight="bold",
)
plt.colorbar(im, ax=ax, shrink=0.6, label="CNV score")
plt.tight_layout()
fig.savefig(FIG_DIR / "cnv_heatmap_chromosome.pdf", dpi=200, bbox_inches="tight")
plt.show()

## Step 8 — CNV UMAP Overlay

Overlaying CNV load on UMAP confirms that annotated tumour clusters have higher chromosomal instability. If annotation and CNV inference agree, both the annotation and the CNV method are validated.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata, color="cnv_load", ax=axes[0], show=False,
           frameon=False, size=2, cmap="YlOrRd", title="CNV load score",
           vmin=0, vmax=adata.obs["cnv_load"].quantile(0.95))
sc.pl.umap(adata, color="cell_type_broad", ax=axes[1], show=False,
           frameon=False, size=2, title="Cell type (broad)",
           legend_loc="right margin")
plt.suptitle("CNV load vs. cell type annotation", fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(FIG_DIR / "cnv_umap_score.pdf", dpi=200, bbox_inches="tight")
plt.show()
print("Tumour clusters should have warm (high) CNV load; stromal/immune near-zero.")

## Step 9 — Summary Tables and Save

In [ ]:
# Per-cell CNV scores
cnv_cols = ["cnv_load", "is_tumour_cnv"] + [
    f"cnv_{r}" for r in RB_SCNA_REGIONS if f"cnv_{r}" in adata.obs.columns
]
adata.obs[cnv_cols].to_csv(TAB_DIR / "cnv_scores_per_cell.csv")

# CNV load by cell type
cnv_summary = adata.obs.groupby("cell_type_broad")["cnv_load"].agg(
    ["mean", "median", "std", "count"]
).round(4)
cnv_summary.to_csv(TAB_DIR / "cnv_load_by_celltype.csv")
print("CNV load by cell type:")
print(cnv_summary.to_string())

# Save
print(f"\nSaving atlas -> {OUT_H5AD.name}")
adata.write_h5ad(OUT_H5AD, compression="gzip")
size_mb = OUT_H5AD.stat().st_size / 1e6
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes | {size_mb:.0f} MB")
print("  Added: .obs['cnv_load'], .obs['is_tumour_cnv'], .obs['cnv_chr*'], .obsm['X_cnv']")
print("  Next: Run notebook 06_subtype_scoring.ipynb")